# 04 · Train with PyTorch Lightning — Single Node

단일 노드 토폴로지(1×1 / 1×N)를 다룹니다. Multi-node(M×N)는 [`05-train_pytorch_lightning_multi_node.ipynb`](05-train_pytorch_lightning_multi_node.ipynb).

| 섹션 | launcher | Trainer |
|------|----------|---------|
| 1×1  | (직접 실행) | `devices=1, num_nodes=1` |
| 1×N  | `TorchDistributor(local_mode=True)` | `devices=N, num_nodes=1, strategy='ddp'` |

데이터셋·모델·하이퍼파라미터는 모든 섹션 공통(`CONFIG`, ML-25M).

1×N에서 Lightning의 `ddp_notebook` 대신 **TorchDistributor**로 worker 프로세스를 띄우는 이유: 1×1 섹션에서 이미 driver의 CUDA가 초기화됐기 때문에 fork 기반 spawn(`ddp_notebook`)이 실패합니다. TorchDistributor는 fresh Python subprocess를 띄우고, 그 안에서 Lightning이 미리 세팅된 `RANK`/`WORLD_SIZE` 환경변수를 그대로 사용합니다.

**클러스터 (이 노트북 전체 공용)**: Single node, DBR 17.3 LTS ML, Driver `g5.12xlarge` (4× A10G), Workers 0, Autoscaling off. 1×1 섹션은 GPU 1개만 사용해도 같은 클러스터에서 그대로 실행.

`MLFlowLogger`는 rank 0에서만 자동 로깅하므로 별도 가드 불필요 ([mlflow-tracking.md](../00-foundations/mlflow-tracking.md)).

In [ ]:
%run ./00-setup

## LightningModule + DataModule

1×1과 1×N 섹션이 그대로 사용합니다.

In [ ]:
import os

import lightning as L
import pyarrow.parquet as pq
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


class TwoTowerLitModule(L.LightningModule):
    def __init__(self, n_users, n_items, emb_dim, tower_hidden, lr=1e-3, weight_decay=1e-5):
        super().__init__()
        self.save_hyperparameters()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        self.user_tower = self._make_tower(emb_dim, tower_hidden)
        self.item_tower = self._make_tower(emb_dim, tower_hidden)
        self.loss_fn = nn.BCEWithLogitsLoss()

    @staticmethod
    def _make_tower(in_dim, hidden):
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        return nn.Sequential(*layers)

    def forward(self, user_ids, item_ids):
        u = self.user_tower(self.user_emb(user_ids))
        i = self.item_tower(self.item_emb(item_ids))
        return (u * i).sum(dim=-1)

    def training_step(self, batch, batch_idx):
        u, i, y = batch
        logits = self(u, i)
        loss = self.loss_fn(logits, y)
        self.log("train/loss", loss, on_step=True, on_epoch=False, prog_bar=True, sync_dist=True)
        return loss

    def validation_step(self, batch, batch_idx):
        u, i, y = batch
        logits = self(u, i)
        loss = self.loss_fn(logits, y)
        # EarlyStopping callback이 monitor하는 키. on_epoch=True로 epoch 평균 집계, sync_dist=True로 DDP 합산.
        self.log("val/loss", loss, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.AdamW(
            self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay
        )


class InteractionsDataModule(L.LightningDataModule):
    def __init__(self, data_dir, batch_size, num_workers=2):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.num_workers = num_workers

    def _load_split(self, split):
        split_dir = os.path.join(self.data_dir, split)
        files = sorted(
            os.path.join(split_dir, f) for f in os.listdir(split_dir) if f.endswith(".parquet")
        )
        table = pq.read_table(files, columns=["user_id", "item_id", "label"])
        return TensorDataset(
            torch.from_numpy(table.column("user_id").to_numpy()),
            torch.from_numpy(table.column("item_id").to_numpy()),
            torch.from_numpy(table.column("label").to_numpy()),
        )

    def setup(self, stage=None):
        self.train_dataset = self._load_split("train")
        self.val_dataset = self._load_split("val")

    def train_dataloader(self):
        # Trainer(strategy='ddp')가 DistributedSampler를 자동 주입한다.
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            pin_memory=True,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True,
        )

## 1×1 GPU

`Trainer(accelerator='gpu', devices=1, num_nodes=1)`. `EarlyStopping(monitor="val/loss", patience=PATIENCE, min_delta=MIN_DELTA)` callback이 epoch 단위 종료를 결정.

In [ ]:
import mlflow
from lightning.pytorch.callbacks import EarlyStopping
from lightning.pytorch.loggers import MLFlowLogger

cfg = CONFIG
model = TwoTowerLitModule(
    n_users=cfg["n_users"],
    n_items=cfg["n_items"],
    emb_dim=cfg["emb_dim"],
    tower_hidden=cfg["tower_hidden"],
)
dm = InteractionsDataModule(
    data_dir=DATA_DIR,
    batch_size=cfg["batch_size_per_gpu"],
)

mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="lightning-1x1", log_system_metrics=True) as run:
    logger = MLFlowLogger(
        experiment_name=EXPERIMENT_PATH,
        tracking_uri="databricks",
        run_id=run.info.run_id,
    )
    early_stop = EarlyStopping(
        monitor="val/loss",
        patience=PATIENCE,
        min_delta=MIN_DELTA,
        mode="min",
    )
    trainer = L.Trainer(
        accelerator="gpu",
        devices=1,
        num_nodes=1,
        max_epochs=cfg["num_epochs"],
        limit_train_batches=cfg["max_steps_per_epoch"],
        logger=logger,
        callbacks=[early_stop],
        default_root_dir=os.path.join(CKPT_DIR, "lightning-1x1"),
    )
    trainer.fit(model, datamodule=dm)

## 1×N GPU

`TorchDistributor(num_processes=N, local_mode=True).run(train_fn)`에서 `Trainer(devices=N, num_nodes=1, strategy='ddp').fit(...)` 호출. **1×1과 같은 `CONFIG`·같은 `DATA_DIR`**. `NUM_GPUS_PER_NODE`를 클러스터 driver의 GPU 수에 맞춥니다.

TorchDistributor child 안에서 LightningModule/DataModule을 다시 정의해 cloudpickle 안전성 확보 ([common-pitfalls.md §2](../00-foundations/common-pitfalls.md)). `EarlyStopping` callback도 child에서 import.

In [ ]:
def train_fn_lightning_1xn(
    experiment_path,
    run_id,
    db_host,
    db_token,
    data_dir,
    ckpt_dir,
    n_users,
    n_items,
    emb_dim,
    tower_hidden,
    batch_size,
    num_epochs,
    max_steps_per_epoch,
    patience,
    min_delta,
    num_gpus_per_node,
):
    import os

    import lightning as L
    import mlflow
    import pyarrow.parquet as pq
    import torch
    import torch.nn as nn
    from lightning.pytorch.callbacks import EarlyStopping
    from lightning.pytorch.loggers import MLFlowLogger
    from torch.utils.data import DataLoader, TensorDataset

    os.environ["DATABRICKS_HOST"] = db_host
    os.environ["DATABRICKS_TOKEN"] = db_token

    class TwoTowerLitModule(L.LightningModule):
        def __init__(self, n_users, n_items, emb_dim, tower_hidden, lr=1e-3, weight_decay=1e-5):
            super().__init__()
            self.save_hyperparameters()
            self.user_emb = nn.Embedding(n_users, emb_dim)
            self.item_emb = nn.Embedding(n_items, emb_dim)
            self.user_tower = self._make_tower(emb_dim, tower_hidden)
            self.item_tower = self._make_tower(emb_dim, tower_hidden)
            self.loss_fn = nn.BCEWithLogitsLoss()

        @staticmethod
        def _make_tower(in_dim, hidden):
            layers, prev = [], in_dim
            for h in hidden:
                layers += [nn.Linear(prev, h), nn.ReLU()]
                prev = h
            return nn.Sequential(*layers)

        def forward(self, user_ids, item_ids):
            u = self.user_tower(self.user_emb(user_ids))
            i = self.item_tower(self.item_emb(item_ids))
            return (u * i).sum(dim=-1)

        def training_step(self, batch, batch_idx):
            u, i, y = batch
            logits = self(u, i)
            loss = self.loss_fn(logits, y)
            self.log("train/loss", loss, on_step=True, sync_dist=True)
            return loss

        def validation_step(self, batch, batch_idx):
            u, i, y = batch
            logits = self(u, i)
            loss = self.loss_fn(logits, y)
            self.log("val/loss", loss, on_step=False, on_epoch=True, sync_dist=True)
            return loss

        def configure_optimizers(self):
            return torch.optim.AdamW(
                self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay
            )

    class InteractionsDataModule(L.LightningDataModule):
        def __init__(self, data_dir, batch_size, num_workers=2):
            super().__init__()
            self.data_dir = data_dir
            self.batch_size = batch_size
            self.num_workers = num_workers

        def _load_split(self, split):
            split_dir = os.path.join(self.data_dir, split)
            files = sorted(
                os.path.join(split_dir, f) for f in os.listdir(split_dir) if f.endswith(".parquet")
            )
            table = pq.read_table(files, columns=["user_id", "item_id", "label"])
            return TensorDataset(
                torch.from_numpy(table.column("user_id").to_numpy()),
                torch.from_numpy(table.column("item_id").to_numpy()),
                torch.from_numpy(table.column("label").to_numpy()),
            )

        def setup(self, stage=None):
            self.train_dataset = self._load_split("train")
            self.val_dataset = self._load_split("val")

        def train_dataloader(self):
            return DataLoader(
                self.train_dataset,
                batch_size=self.batch_size,
                shuffle=True,
                num_workers=self.num_workers,
                pin_memory=True,
            )

        def val_dataloader(self):
            return DataLoader(
                self.val_dataset,
                batch_size=self.batch_size,
                shuffle=False,
                num_workers=self.num_workers,
                pin_memory=True,
            )

    model = TwoTowerLitModule(
        n_users=n_users, n_items=n_items, emb_dim=emb_dim, tower_hidden=tower_hidden
    )
    dm = InteractionsDataModule(data_dir=data_dir, batch_size=batch_size)

    logger = MLFlowLogger(
        experiment_name=experiment_path,
        tracking_uri="databricks",
        run_id=run_id,
    )
    early_stop = EarlyStopping(
        monitor="val/loss",
        patience=patience,
        min_delta=min_delta,
        mode="min",
    )
    trainer = L.Trainer(
        accelerator="gpu",
        devices=num_gpus_per_node,
        num_nodes=1,
        strategy="ddp",
        max_epochs=num_epochs,
        limit_train_batches=max_steps_per_epoch,
        logger=logger,
        callbacks=[early_stop],
        default_root_dir=ckpt_dir,
    )

    rank = int(os.environ.get("RANK", "0"))
    if rank == 0:
        # 같은 worker 프로세스에서 driver 측 run에 attach해 system metrics를 워커 노드 기준으로 수집한다.
        mlflow.start_run(run_id=run_id, log_system_metrics=True)
    try:
        trainer.fit(model, datamodule=dm)
    finally:
        if rank == 0 and mlflow.active_run() is not None:
            mlflow.end_run()
    return "ok"


import mlflow
from pyspark.ml.torch.distributor import TorchDistributor

NUM_GPUS_PER_NODE = 4  # 클러스터 driver GPU 수

cfg = CONFIG
mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="lightning-1xN", log_system_metrics=True) as run:
    run_id = run.info.run_id

TorchDistributor(
    num_processes=NUM_GPUS_PER_NODE,
    local_mode=True,
    use_gpu=True,
).run(
    train_fn_lightning_1xn,
    experiment_path=EXPERIMENT_PATH,
    run_id=run_id,
    db_host=DB_HOST,
    db_token=DB_TOKEN,
    data_dir=DATA_DIR,
    ckpt_dir=os.path.join(CKPT_DIR, "lightning-1xN"),
    n_users=cfg["n_users"],
    n_items=cfg["n_items"],
    emb_dim=cfg["emb_dim"],
    tower_hidden=cfg["tower_hidden"],
    batch_size=cfg["batch_size_per_gpu"],
    num_epochs=cfg["num_epochs"],
    max_steps_per_epoch=cfg["max_steps_per_epoch"],
    patience=PATIENCE,
    min_delta=MIN_DELTA,
    num_gpus_per_node=NUM_GPUS_PER_NODE,
)